# 3. Merging

Merging joins two branches back together. There are three cases:

1. **Fast-forward** — nothing to join, git just moves the branch forward.
2. **Merge commit** — both branches changed, git combines them.
3. **Conflict** — both branches changed the same line, and you decide.

We start from three branches, each changing a different part of the file:

```mermaid
gitGraph
    commit id: "volume"
    branch froude
    commit id: "froude number"
    checkout main
    branch report
    commit id: "units"
    checkout main
    branch fresh-water
    commit id: "fresh water"
```

In [ ]:
!rm -rf ~/git-course/03-merging
!mkdir -p ~/git-course/03-merging
%cd ~/git-course/03-merging

## Starting point

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

Branch `froude` adds a function at the end of the file:

In [ ]:
!git switch -c froude

In [ ]:
%%writefile -a hydrostatics.py


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

In [ ]:
!git add hydrostatics.py
!git commit -m "Add the Froude number"

Branch `report` changes the first line. `git switch -c report main` means
"create `report`, starting from `main`".

In [ ]:
!git switch -c report main

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git add hydrostatics.py
!git commit -m "Say which units the module uses"

Branch `fresh-water` changes the density:

In [ ]:
!git switch -c fresh-water main

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1000.0  # kg/m^3, fresh water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git add hydrostatics.py
!git commit -m "Use fresh water density for river barges"

In [ ]:
!git switch main
!git log --oneline --graph --all

## Case 1: fast-forward

Nothing has happened on `main` since `froude` started. So git does not
need to combine anything — it simply moves `main` forward.

In [ ]:
!git merge froude

In [ ]:
!git log --oneline --graph --all

`main` and `froude` now point at the same commit. The branch has done its
job:

In [ ]:
!git branch -d froude

## Case 2: a merge commit

First, `main` gets a new commit of its own:

In [ ]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

In [ ]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

Now `main` and `report` have both moved. Git combines them in a new commit
with **two parents**:

```mermaid
gitGraph
    commit id: "volume"
    commit id: "froude number"
    branch report
    commit id: "units"
    checkout main
    commit id: "tonnes per cm"
    merge report
```

In [ ]:
!git merge report

In [ ]:
!git log --oneline --graph --all

The two branches changed different lines, so git merged them by itself.
Most merges are like this.

In [ ]:
!cat hydrostatics.py

In [ ]:
!git branch -d report

## Case 3: a conflict

On `main`, someone sets the density to the ITTC standard value, 1026:

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1026.0  # kg/m^3, sea water (ITTC standard)


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

In [ ]:
!git add hydrostatics.py
!git commit -m "Use the ITTC standard sea water density"

But `fresh-water` changed the same line to 1000. Git cannot choose for you:

In [ ]:
# This one fails on purpose: a conflict.
!git merge fresh-water

Nothing is broken. The merge is paused, and `git status` says so:

In [ ]:
!git status

Git wrote both versions into the file:

In [ ]:
!cat hydrostatics.py

- Between `<<<<<<< HEAD` and `=======` is **our** version (`main`).
- Between `=======` and `>>>>>>> fresh-water` is **their** version.

### The escape hatch

Not ready to decide? `git merge --abort` undoes the whole merge:

In [ ]:
!git merge --abort
!git status

Back to normal. Now let's do it for real:

In [ ]:
# This one fails on purpose: the same conflict again.
!git merge fresh-water

### Resolving

Edit the file so it is right, and remove the marker lines. You don't have
to pick a side — here we keep both densities:

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO_SEA = 1026.0  # kg/m^3, sea water (ITTC standard)
RHO_FRESH = 1000.0  # kg/m^3, fresh water
RHO = RHO_SEA


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

`git add` marks the file as resolved, and `git commit` finishes the merge.
`--no-edit` keeps the message git already prepared.

In [ ]:
!git add hydrostatics.py
!git commit --no-edit

In [ ]:
!git log --oneline --graph --all

In [ ]:
!git branch -d fresh-water

## Summary

| I want to… | Command |
| - | - |
| merge a branch into the current one | `git merge <branch>` |
| give up on a merge | `git merge --abort` |
| mark a conflict as resolved | `git add <file>` |
| finish the merge | `git commit --no-edit` |

Merge often: small merges rarely conflict.

## Exercises

1. Make two branches that change the same line, merge them, and resolve
   the conflict.
2. Start a merge with a conflict and get out of it with `git merge --abort`.

Next: [4. Rebase](04-rebase.ipynb)